In [1]:
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")                      # headless-safe; figures are saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GridSearchCV, GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# --------------------------------------------------------------------------- #
# 0. CONFIGURATION
# --------------------------------------------------------------------------- #
DATA_DIR = Path(".")                       # folder containing the 7 OULAD CSVs
OUT_DIR = Path("outputs/oulad")
FIG_DIR, TAB_DIR, CACHE_DIR = OUT_DIR / "figures", OUT_DIR / "tables", OUT_DIR / "cache"
for d in (FIG_DIR, TAB_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
EARLY_WEEKS = 8                 # main prediction window: first 8 weeks (days 0-55)
CUTOFF_WEEKS = [2, 4, 8, 12, 16, 24, None]   # early-warning curve; None = whole course
CHUNK_SIZE = 2_000_000          # rows per chunk when streaming studentVle.csv
W_MIN, W_MAX = -4, 39           # week range plotted (negative = before course start)
KEYS = ["code_module", "code_presentation", "id_student"]   # one row = one enrolment
TOP_ACTIVITY_TYPES = ["oucontent", "forumng", "homepage", "quiz", "resource", "subpage", "url"]

sns.set_theme(style="whitegrid", context="notebook")
OUTCOME_ORDER = ["Distinction", "Pass", "Fail", "Withdrawn"]
OUTCOME_PALETTE = dict(zip(OUTCOME_ORDER, sns.color_palette("colorblind", 4)))

In [3]:
# --------------------------------------------------------------------------- #
# 1. DATA LOADING
# --------------------------------------------------------------------------- #
def load_small_tables():
    """Load the six small OULAD tables and fix dtypes."""
    t = {name: pd.read_csv(DATA_DIR / f"{name}.csv") for name in
         ["courses", "assessments", "vle", "studentInfo", "studentRegistration",
          "studentAssessment"]}
    return t


def key_dtypes(courses):
    """Shared categorical dtypes so merges/groupbys on module & presentation are fast."""
    return {"code_module": pd.CategoricalDtype(sorted(courses.code_module.unique())),
            "code_presentation": pd.CategoricalDtype(sorted(courses.code_presentation.unique()))}


def as_cat(df, dtypes):
    df = df.copy()
    for col, dt in dtypes.items():
        if col in df.columns:
            df[col] = df[col].astype(dt)
    return df


def aggregate_student_vle(path, vle, dtypes):
    """
    Stream the (very large, ~10M rows) studentVle.csv in chunks and compress it into two
    compact tables so the rest of the pipeline fits in memory:
        daily        : one row per (enrolment, day)            -> summed clicks
        weekly_type  : one row per (enrolment, week, activity)  -> summed clicks
    Results are cached to disk so the big file is only read once.
    """
    f_daily, f_weekly = CACHE_DIR / "daily.pkl", CACHE_DIR / "weekly_type.pkl"
    if f_daily.exists() and f_weekly.exists():
        print("  using cached aggregates of studentVle.csv")
        return pd.read_pickle(f_daily), pd.read_pickle(f_weekly)

    site_type = vle.set_index("id_site")["activity_type"]   # id_site is unique in vle.csv
    daily_parts, weekly_parts = [], []
    reader = pd.read_csv(path, chunksize=CHUNK_SIZE,
                         usecols=["code_module", "code_presentation", "id_student",
                                  "id_site", "date", "sum_click"],
                         dtype={"id_student": "int32", "id_site": "int32",
                                "date": "int16", "sum_click": "int32"})
    for i, ch in enumerate(reader):
        ch = as_cat(ch, dtypes)
        ch["activity_type"] = ch["id_site"].map(site_type).fillna("unknown")
        ch["week"] = np.floor_divide(ch["date"], 7).astype("int16")
        daily_parts.append(ch.groupby(KEYS + ["date"], observed=True)["sum_click"]
                             .sum().reset_index())
        weekly_parts.append(ch.groupby(KEYS + ["week", "activity_type"], observed=True)
                              ["sum_click"].sum().reset_index())
        print(f"  chunk {i + 1} processed")
    # a (student, day) can straddle two chunks -> re-aggregate after concatenation
    daily = (pd.concat(daily_parts).groupby(KEYS + ["date"], observed=True)["sum_click"]
             .sum().reset_index().sort_values(KEYS + ["date"]).reset_index(drop=True))
    weekly_type = (pd.concat(weekly_parts)
                   .groupby(KEYS + ["week", "activity_type"], observed=True)["sum_click"]
                   .sum().reset_index())
    daily.to_pickle(f_daily)
    weekly_type.to_pickle(f_weekly)
    return daily, weekly_type

In [4]:
# --------------------------------------------------------------------------- #
# 2. MERGING & STUDENT-LEVEL TABLE
# --------------------------------------------------------------------------- #
def build_student_table(info, reg, courses, dtypes):
    """
    One row per enrolment (module x presentation x student) with demographics,
    registration timing and the outcome label.
    NOTE: id_student alone is NOT a unique key - the same learner can take several
    modules/presentations - so all joins use the 3-column KEYS.
    """
    st = (as_cat(info, dtypes)
          .merge(as_cat(reg, dtypes), on=KEYS, how="left", validate="one_to_one")
          .merge(as_cat(courses, dtypes), on=["code_module", "code_presentation"], how="left"))
    st["imd_band"] = st["imd_band"].fillna("Unknown")          # 1,111 missing -> own category
    st["passed"] = st["final_result"].isin(["Pass", "Distinction"]).astype(int)
    # Binary target: 1 = Pass/Distinction, 0 = Fail/Withdrawn  (see report, Section 2)
    return st.reset_index(drop=True)

In [5]:
# --------------------------------------------------------------------------- #
# 3. FEATURE ENGINEERING
# --------------------------------------------------------------------------- #
def week_matrix(weekly_total, st, w_min, w_max, mask_unreg=False):
    """
    Students x weeks matrix of total clicks (float32). Weeks that lie beyond a module's
    length are NaN. If mask_unreg, weeks after the student withdrew are NaN as well
    (used for plots so that dropouts do not drag the group means down artificially).
    """
    nw = w_max - w_min + 1
    W = np.zeros((len(st), nw), dtype=np.float32)
    rows = st[KEYS].reset_index().rename(columns={"index": "row"})
    m = weekly_total.merge(rows, on=KEYS, how="inner")
    m = m[(m.week >= w_min) & (m.week <= w_max)]
    np.add.at(W, (m["row"].values, (m["week"].values - w_min)), m["sum_click"].values)
    week_start = (np.arange(w_min, w_max + 1) * 7)[None, :]
    W[week_start > st["module_presentation_length"].values[:, None]] = np.nan
    if mask_unreg:
        unreg = st["date_unregistration"].values[:, None]
        W[(~np.isnan(unreg)) & (week_start >= unreg)] = np.nan
    return W


def vle_features(st, daily, weekly_total, weekly_type, cut_weeks):
    """Clickstream features restricted to the first `cut_weeks` weeks (None = all)."""
    cut_day = None if cut_weeks is None else cut_weeks * 7 - 1
    d = daily if cut_day is None else daily[daily["date"] <= cut_day]
    inc = d[d["date"] >= 0]                                   # in-course days only
    g = inc.groupby(KEYS, observed=True)

    f = pd.DataFrame({
        "total_clicks": g["sum_click"].sum(),
        "active_days": g["date"].nunique(),
        "last_active_day": g["date"].max(),
    })
    f["clicks_per_active_day"] = f["total_clicks"] / f["active_days"]
    # longest run of consecutive inactive days between two active days
    gap = inc.groupby(KEYS, observed=True)["date"].diff()
    f["longest_gap_days"] = gap.groupby([inc[k] for k in KEYS], observed=True).max()
    f["first_access_day"] = d.groupby(KEYS, observed=True)["date"].min()  # <0 = before start
    pre = d[d["date"] < 0].groupby(KEYS, observed=True)["sum_click"].sum().rename("pre_course_clicks")
    f = f.join(pre, how="outer").reset_index()
    out = st.merge(f, on=KEYS, how="left")

    # students with no click at all
    out["zero_activity"] = out["total_clicks"].isna().astype(int)
    for c in ["total_clicks", "active_days", "pre_course_clicks", "clicks_per_active_day"]:
        out[c] = out[c].fillna(0)
    out["longest_gap_days"] = out["longest_gap_days"].fillna(0)
    end_day = out["module_presentation_length"] if cut_day is None else \
        np.minimum(cut_day, out["module_presentation_length"])
    out["days_since_last_active"] = (end_day - out["last_active_day"]).fillna(end_day + 1).clip(lower=0)
    out["first_access_day"] = out["first_access_day"].fillna(end_day)    # never accessed -> end
    out = out.drop(columns="last_active_day")

    # ---- weekly-pattern features (self-regulation / consistency) ----
    w_max = (cut_weeks - 1) if cut_weeks is not None else W_MAX
    W = week_matrix(weekly_total, out, 0, w_max)
    valid = ~np.isnan(W)
    Wz = np.where(valid, W, 0.0)
    n = valid.sum(1)
    mean = Wz.sum(1) / n
    sd = np.sqrt((valid * (Wz - mean[:, None]) ** 2).sum(1) / n)
    x = np.broadcast_to(np.arange(W.shape[1], dtype=float), W.shape)
    xm = (x * valid).sum(1) / n
    cov = (valid * (x - xm[:, None]) * (Wz - mean[:, None])).sum(1)
    var = (valid * (x - xm[:, None]) ** 2).sum(1)
    with np.errstate(divide="ignore", invalid="ignore"):
        out["mean_weekly_clicks"] = mean
        out["weekly_cv"] = np.where(mean > 0, sd / mean, 0.0)       # burstiness (0 if no clicks)
        out["active_week_ratio"] = ((Wz > 0) & valid).sum(1) / n
        out["trend_slope"] = np.where((mean > 0) & (var > 0), cov / var / mean, 0.0)  # relative slope

    # ---- activity-type mix (share of clicks) ----
    wt = weekly_type[weekly_type["week"] >= 0]
    if cut_weeks is not None:
        wt = wt[wt["week"] < cut_weeks]
    wt = wt.assign(activity_type=np.where(wt["activity_type"].isin(TOP_ACTIVITY_TYPES),
                                          wt["activity_type"], "other"))
    mix = (wt.groupby(KEYS + ["activity_type"], observed=True)["sum_click"].sum()
             .unstack(fill_value=0))
    mix = mix.div(mix.sum(axis=1), axis=0).add_prefix("share_").reset_index()
    out = out.merge(mix, on=KEYS, how="left")
    share_cols = [c for c in out.columns if c.startswith("share_")]
    out[share_cols] = out[share_cols].fillna(0)
    return out


def assessment_features(st, assessments, student_assessment, cut_weeks):
    """
    Submission behaviour for TMA/CMA assessments (exams excluded: no deadline in the log
    and they happen at the end of the course). Only assessments whose deadline AND the
    student's submission fall inside the window are used -> no look-ahead leakage.
    latency = date_submitted - deadline (days). Negative = submitted early.
    """
    cut_day = None if cut_weeks is None else cut_weeks * 7 - 1
    a = assessments[assessments["assessment_type"] != "Exam"].copy()
    a = a.rename(columns={"date": "deadline"})
    if cut_day is not None:
        a = a[a["deadline"] <= cut_day]
    n_due = a.groupby(["code_module", "code_presentation"]).size().rename("n_due").reset_index()

    sa = student_assessment.merge(a, on="id_assessment", how="inner")
    if cut_day is not None:
        sa = sa[sa["date_submitted"] <= cut_day]
    sa["latency"] = sa["date_submitted"] - sa["deadline"]
    sa["late"] = (sa["latency"] > 0).astype(int)
    agg = (sa.groupby(["code_module", "code_presentation", "id_student"])
             .agg(n_submitted=("id_assessment", "nunique"), mean_score=("score", "mean"),
                  mean_latency=("latency", "mean"), share_late=("late", "mean"))
             .reset_index())
    agg = as_cat(agg, dict(st[["code_module", "code_presentation"]].dtypes))
    n_due = as_cat(n_due, dict(st[["code_module", "code_presentation"]].dtypes))

    out = (st.merge(agg, on=KEYS, how="left")
             .merge(n_due, on=["code_module", "code_presentation"], how="left"))
    out["n_due"] = out["n_due"].fillna(0)
    out["n_submitted"] = out["n_submitted"].fillna(0)
    out["no_submissions"] = (out["n_submitted"] == 0).astype(int)
    out["submission_rate"] = np.where(out["n_due"] > 0,
                                      (out["n_submitted"] / out["n_due"]).clip(upper=1), 0.0)
    return out[KEYS + ["n_submitted", "n_due", "submission_rate", "mean_score",
                       "mean_latency", "share_late", "no_submissions"]]


def make_model_table(st, daily, weekly_total, weekly_type, assessments, student_assessment,
                     cut_weeks):
    """Full modelling table for a given prediction window."""
    cut_day = None if cut_weeks is None else cut_weeks * 7 - 1
    pop = st
    if cut_day is not None:
        # students who already un-registered before the cut-off are known drop-outs -> not predicted
        pop = st[st["date_unregistration"].isna() | (st["date_unregistration"] > cut_day)] \
            .reset_index(drop=True)
    df = vle_features(pop, daily, weekly_total, weekly_type, cut_weeks)
    df = df.merge(assessment_features(pop, assessments, student_assessment, cut_weeks),
                  on=KEYS, how="left")
    df["log_total_clicks"] = np.log1p(df["total_clicks"])
    df["registration_lead_days"] = -df["date_registration"]    # larger = registered earlier
    df["code_module"] = df["code_module"].astype(str)
    df["code_presentation"] = df["code_presentation"].astype(str)
    return df


In [6]:
# --------------------------------------------------------------------------- #
# 4. TEMPORAL VISUALISATIONS
# --------------------------------------------------------------------------- #
def mean_ci_by_group(W, groups, order, min_n=30):
    """Mean +/- 95% CI per week for each group (NaN-aware, hides weeks with < min_n students)."""
    weeks = np.arange(W_MIN, W_MAX + 1)
    res = {}
    for gname in order:
        sub = W[(groups == gname).values]
        n = (~np.isnan(sub)).sum(0)
        mean = np.nanmean(sub, axis=0)
        ci = 1.96 * np.nanstd(sub, axis=0, ddof=1) / np.sqrt(np.maximum(n, 1))
        mean[n < min_n] = np.nan
        res[gname] = (weeks, mean, ci)
    return res


def plot_weekly_by_group(W, st, group_col, order, palette, fname, title):
    fig, ax = plt.subplots(figsize=(10, 5))
    for gname, (weeks, mean, ci) in mean_ci_by_group(W, st[group_col], order).items():
        ax.plot(weeks, mean, label=gname, color=palette[gname], lw=2)
        ax.fill_between(weeks, mean - ci, mean + ci, color=palette[gname], alpha=0.15)
    ax.axvline(0, color="grey", ls="--", lw=1)
    ax.set(xlabel="Week relative to course start", ylabel="Mean VLE clicks per student",
           title=title)
    ax.legend(title=group_col)
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=200)
    plt.close(fig)


def plot_weekly_by_module(W, st, fname):
    mods = sorted(st["code_module"].astype(str).unique())
    fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharex=True)
    for ax, mod in zip(axes.ravel(), mods):
        m = (st["code_module"].astype(str) == mod).values
        for gname, (weeks, mean, ci) in mean_ci_by_group(W[m], st.loc[m, "final_result"],
                                                         OUTCOME_ORDER, min_n=15).items():
            ax.plot(weeks, mean, color=OUTCOME_PALETTE[gname], lw=1.6, label=gname)
        ax.set_title(f"Module {mod}")
        ax.axvline(0, color="grey", ls="--", lw=0.8)
    for ax in axes.ravel()[len(mods):]:
        ax.axis("off")
    axes[0, 0].legend(fontsize=8)
    fig.supxlabel("Week relative to course start")
    fig.supylabel("Mean clicks per student")
    fig.suptitle("Weekly engagement trajectories by final result, per module")
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=200)
    plt.close(fig)


def plot_activity_mix(df, fname):
    share_cols = [c for c in df.columns if c.startswith("share_")]
    mix = df.groupby("final_result")[share_cols].mean().loc[OUTCOME_ORDER]
    mix.columns = [c.replace("share_", "") for c in mix.columns]
    ax = mix.plot(kind="barh", stacked=True, figsize=(10, 4.5), colormap="tab20")
    ax.set(xlabel="Mean share of clicks", ylabel="",
           title=f"Activity-type mix in the first {EARLY_WEEKS} weeks, by final result")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=200)
    plt.close()


def plot_consistency_heatmap(df, fname):
    """H1: pass rate by click-volume quartile x burstiness tertile (both within module)."""
    d = df[df["total_clicks"] > 0].copy()
    d["volume_q"] = d.groupby("code_module")["total_clicks"].transform(
        lambda s: pd.qcut(s.rank(method="first"), 4, labels=["Q1 low", "Q2", "Q3", "Q4 high"]))
    d["regularity"] = d.groupby("code_module")["weekly_cv"].transform(
        lambda s: pd.qcut(s.rank(method="first"), 3, labels=["steady", "mixed", "bursty"]))
    pr = d.pivot_table(index="volume_q", columns="regularity", values="passed",
                       aggfunc="mean", observed=True)
    cnt = d.pivot_table(index="volume_q", columns="regularity", values="passed",
                        aggfunc="size", observed=True)
    pr.to_csv(TAB_DIR / "h1_passrate_volume_by_regularity.csv")
    annot = pr.round(2).astype(str) + "\n(n=" + cnt.astype(str) + ")"
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(pr, annot=annot, fmt="", cmap="YlGnBu", vmin=0, vmax=1, ax=ax,
                cbar_kws={"label": "Pass rate"})
    ax.set(title="H1: pass rate by click volume and weekly regularity", xlabel="", ylabel="")
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=200)
    plt.close(fig)


def plot_latency(df, fname):
    d = df.dropna(subset=["mean_latency"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    for g in OUTCOME_ORDER:
        x = np.sort(d.loc[d["final_result"] == g, "mean_latency"].values)
        if len(x):
            axes[0].plot(x, np.arange(1, len(x) + 1) / len(x), color=OUTCOME_PALETTE[g], label=g)
    axes[0].axvline(0, color="grey", ls="--")
    axes[0].set(xlim=(-30, 30), xlabel="Mean submission latency (days; <0 = before deadline)",
                ylabel="ECDF", title="Submission latency, first 8 weeks")
    axes[0].legend()
    sns.boxplot(data=df, x="final_result", y="submission_rate", order=OUTCOME_ORDER,
                palette=OUTCOME_PALETTE, ax=axes[1])
    axes[1].set(xlabel="", ylabel="Share of due assessments submitted",
                title="Submission rate, first 8 weeks")
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=200)
    plt.close(fig)


In [7]:
# --------------------------------------------------------------------------- #
# 5. MACHINE LEARNING
# --------------------------------------------------------------------------- #
DEMO_NUM = ["num_of_prev_attempts", "studied_credits"]
DEMO_CAT = ["gender", "region", "highest_education", "imd_band", "age_band", "disability"]
CONTEXT_CAT = ["code_module", "code_presentation"]
CAT_COLS = set(DEMO_CAT + CONTEXT_CAT)
SHARES = ["share_" + t for t in TOP_ACTIVITY_TYPES + ["other"]]
VOLUME = ["total_clicks"]
CONSISTENCY = ["active_days", "active_week_ratio", "weekly_cv", "longest_gap_days",
               "days_since_last_active", "trend_slope", "zero_activity"]
EARLINESS = ["pre_course_clicks", "first_access_day", "registration_lead_days"]
ASSESS = ["n_submitted", "submission_rate", "mean_score", "mean_latency", "share_late",
          "no_submissions"]
FEATURE_SETS = {                                    # used for the ablation study (H1 & H2)
    "Demographics only": DEMO_NUM + DEMO_CAT + CONTEXT_CAT,
    "Click volume only": CONTEXT_CAT + VOLUME,
    "Volume + consistency (H1)": CONTEXT_CAT + VOLUME + CONSISTENCY,
    "Volume + consistency + mix + earliness": CONTEXT_CAT + VOLUME + CONSISTENCY + SHARES + EARLINESS,
    "Assessment timeliness only (H2)": CONTEXT_CAT + ASSESS,
    "All behaviour (VLE + assessment)": CONTEXT_CAT + VOLUME + CONSISTENCY + SHARES + EARLINESS + ASSESS,
    "All features (behaviour + demographics)": DEMO_NUM + DEMO_CAT + CONTEXT_CAT + VOLUME + CONSISTENCY
                                               + SHARES + EARLINESS + ASSESS,
}


def make_preprocessor(cols, scale=False):
    """Median-impute + (optional) scale numerics; one-hot encode categoricals."""
    cat = [c for c in cols if c in CAT_COLS]
    num = [c for c in cols if c not in CAT_COLS]
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    return ColumnTransformer(
        [("num", Pipeline(num_steps), num),
         ("cat", OneHotEncoder(handle_unknown="ignore"), cat)],   # one-hot: gender, region, ...
        sparse_threshold=0.0)


def make_rf(cols, **kw):
    params = dict(n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample",
                  n_jobs=-1, random_state=SEED)
    params.update(kw)
    return Pipeline([("prep", make_preprocessor(cols)), ("clf", RandomForestClassifier(**params))])


def make_lr(cols):
    return Pipeline([("prep", make_preprocessor(cols, scale=True)),
                     ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])


def split_by_student(df, test_size=0.2):
    """Group split: the same learner never appears in both train and test (no identity leakage)."""
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    tr, te = next(gss.split(df, df["passed"], groups=df["id_student"]))
    return df.iloc[tr], df.iloc[te]


def metrics_row(y, pred, proba):
    return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "precision": precision_score(y, pred), "recall": recall_score(y, pred),
            "f1": f1_score(y, pred), "roc_auc": roc_auc_score(y, proba)}


def run_main_models(df):
    """Random Forest (tuned) vs Logistic Regression on the early-window table."""
    all_cols = FEATURE_SETS["All features (behaviour + demographics)"]
    train, test = split_by_student(df)
    Xtr, ytr, Xte, yte = train[all_cols], train["passed"], test[all_cols], test["passed"]
    print(f"  train={len(train):,}  test={len(test):,}  pass rate={df['passed'].mean():.2%}")

    # --- Random Forest with a small grid search, grouped & stratified CV on the TRAIN set only
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    grid = GridSearchCV(make_rf(all_cols, n_estimators=200),
                        {"clf__max_depth": [None, 12], "clf__min_samples_leaf": [1, 3, 8],
                         "clf__max_features": ["sqrt", 0.3]},
                        scoring="roc_auc", cv=cv, n_jobs=1, refit=True)
    grid.fit(Xtr, ytr, groups=train["id_student"])
    rf = grid.best_estimator_
    print("  best RF params:", grid.best_params_, f"| CV AUC={grid.best_score_:.3f}")
    pd.DataFrame(grid.cv_results_).to_csv(TAB_DIR / "rf_gridsearch_results.csv", index=False)

    lr = make_lr(all_cols).fit(Xtr, ytr)

    rows = {}
    fig_roc, ax_roc = plt.subplots(figsize=(6, 5.5))
    for name, model in [("Random Forest", rf), ("Logistic Regression", lr)]:
        proba, pred = model.predict_proba(Xte)[:, 1], model.predict(Xte)
        rows[name] = metrics_row(yte, pred, proba)
        fpr, tpr, _ = roc_curve(yte, proba)
        ax_roc.plot(fpr, tpr, label=f"{name} (AUC={rows[name]['roc_auc']:.3f})")
        fig, ax = plt.subplots(figsize=(4.8, 4.2))
        ConfusionMatrixDisplay(confusion_matrix(yte, pred),
                               display_labels=["Fail/Withdrawn", "Pass/Distinction"]
                               ).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(f"{name}: confusion matrix (test set)")
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"confusion_matrix_{name.split()[0].lower()}.png", dpi=200)
        plt.close(fig)
    ax_roc.plot([0, 1], [0, 1], "k--", lw=1)
    ax_roc.set(xlabel="False positive rate", ylabel="True positive rate",
               title=f"ROC curves, first {EARLY_WEEKS} weeks (test set)")
    ax_roc.legend()
    fig_roc.tight_layout()
    fig_roc.savefig(FIG_DIR / "roc_curves.png", dpi=200)
    plt.close(fig_roc)
    res = pd.DataFrame(rows).T.round(4)
    res.to_csv(TAB_DIR / "main_model_metrics.csv")
    print(res)

    # --- Feature importance: impurity-based (fast) and permutation-based (more reliable)
    names = rf.named_steps["prep"].get_feature_names_out()
    imp = pd.Series(rf.named_steps["clf"].feature_importances_, index=names).sort_values(ascending=False)
    imp.head(25).to_csv(TAB_DIR / "rf_impurity_importance_top25.csv")
    perm = permutation_importance(rf, Xte, yte, scoring="roc_auc", n_repeats=5,
                                  random_state=SEED, n_jobs=1)
    perm = pd.DataFrame({"mean": perm.importances_mean, "std": perm.importances_std},
                        index=all_cols).sort_values("mean", ascending=False)
    perm.to_csv(TAB_DIR / "rf_permutation_importance.csv")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    imp.head(15)[::-1].plot(kind="barh", ax=axes[0], color="steelblue")
    axes[0].set_title("Impurity importance (top 15, one-hot level)")
    perm.head(15)["mean"][::-1].plot(kind="barh", xerr=perm.head(15)["std"][::-1], ax=axes[1],
                                     color="darkorange")
    axes[1].set_title("Permutation importance on test set (drop in AUC)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "feature_importance.png", dpi=200)
    plt.close(fig)

    # --- Standardised logistic coefficients (direction of effects)
    lr_names = lr.named_steps["prep"].get_feature_names_out()
    coef = pd.Series(lr.named_steps["clf"].coef_[0], index=lr_names)
    coef.reindex(coef.abs().sort_values(ascending=False).index).head(25) \
        .to_csv(TAB_DIR / "lr_top_coefficients.csv")
    return rf, train, test


def run_ablation(df):
    """Same split, same RF settings, different feature sets -> tests H1 and H2."""
    train, test = split_by_student(df)
    rows = []
    for name, cols in FEATURE_SETS.items():
        m = make_rf(cols, min_samples_leaf=3).fit(train[cols], train["passed"])
        r = metrics_row(test["passed"], m.predict(test[cols]), m.predict_proba(test[cols])[:, 1])
        rows.append({"feature_set": name, "n_features": len(cols), **r})
    res = pd.DataFrame(rows).round(4)
    res.to_csv(TAB_DIR / "ablation_results.csv", index=False)
    print(res[["feature_set", "accuracy", "f1", "roc_auc"]])
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(data=res, y="feature_set", x="roc_auc", color="steelblue", ax=ax)
    for i, v in enumerate(res["roc_auc"]):
        ax.text(v + 0.003, i, f"{v:.3f}", va="center")
    ax.set(xlim=(0.5, 1), xlabel="Test ROC-AUC", ylabel="",
           title=f"Ablation: which behaviours predict passing? (first {EARLY_WEEKS} weeks)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "ablation_auc.png", dpi=200)
    plt.close(fig)


def run_early_warning_curve(st, daily, weekly_total, weekly_type, assessments, student_assessment):
    """How early can we predict the outcome? Re-train with 2, 4, ..., full-course windows."""
    all_cols = FEATURE_SETS["All features (behaviour + demographics)"]
    rows = []
    for cw in CUTOFF_WEEKS:
        d = make_model_table(st, daily, weekly_total, weekly_type, assessments,
                             student_assessment, cw)
        tr, te = split_by_student(d)
        m = make_rf(all_cols, min_samples_leaf=3).fit(tr[all_cols], tr["passed"])
        r = metrics_row(te["passed"], m.predict(te[all_cols]), m.predict_proba(te[all_cols])[:, 1])
        rows.append({"window_weeks": cw if cw is not None else 40, "label": cw or "full",
                     "n_students": len(d), **r})
        print(f"  window={cw}: AUC={r['roc_auc']:.3f}  F1={r['f1']:.3f}  n={len(d):,}")
    res = pd.DataFrame(rows).round(4)
    res.to_csv(TAB_DIR / "early_warning_curve.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.plot(res["window_weeks"], res["roc_auc"], "o-", label="ROC-AUC")
    ax.plot(res["window_weeks"], res["f1"], "s-", label="F1")
    ax.set(xlabel="Weeks of data available (40 = whole course)", ylabel="Test score",
           title="Early-warning curve: prediction quality vs. observation window")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "early_warning_curve.png", dpi=200)
    plt.close(fig)


def run_hypothesis_tests(df):
    """Univariate Mann-Whitney U tests (Pass/Distinction vs Fail/Withdrawn) + adjusted logit."""
    feats = ["total_clicks", "weekly_cv", "active_week_ratio", "longest_gap_days",
             "days_since_last_active", "pre_course_clicks", "mean_latency", "submission_rate",
             "share_late", "mean_score"]
    rows = []
    for f in feats:
        a, b = df.loc[df.passed == 1, f].dropna(), df.loc[df.passed == 0, f].dropna()
        u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        rows.append({"feature": f, "median_pass": a.median(), "median_notpass": b.median(),
                     "prob_superiority": u / (len(a) * len(b)),   # 0.5 = no effect
                     "p_value": p, "n_pass": len(a), "n_notpass": len(b)})
    pd.DataFrame(rows).round(4).to_csv(TAB_DIR / "hypothesis_mannwhitney.csv", index=False)

    # Adjusted logistic regression: does consistency matter AFTER controlling for volume?
    try:
        import statsmodels.formula.api as smf
        d = df.copy()
        for c in ["weekly_cv", "active_week_ratio", "longest_gap_days", "mean_latency",
                  "submission_rate", "log_total_clicks"]:
            d[c + "_z"] = (d[c] - d[c].mean()) / d[c].std()
        d["mean_latency_z"] = d["mean_latency_z"].fillna(0)
        mod = smf.logit("passed ~ log_total_clicks_z + weekly_cv_z + active_week_ratio_z + "
                        "longest_gap_days_z + mean_latency_z + submission_rate_z + C(code_module)",
                        data=d).fit(disp=0)
        out = pd.DataFrame({"coef": mod.params, "odds_ratio": np.exp(mod.params),
                            "p_value": mod.pvalues}).round(4)
        out.to_csv(TAB_DIR / "h1_h2_adjusted_logit.csv")
        print(out)
    except ImportError:
        print("  (statsmodels not installed - skipping adjusted logit; "
              "use the LR coefficients in lr_top_coefficients.csv instead)")


In [13]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/")

In [14]:
# --------------------------------------------------------------------------- #
# 6. MAIN
# --------------------------------------------------------------------------- #
def main():
    print("[1/6] Loading tables ...")
    t = load_small_tables()
    dtypes = key_dtypes(t["courses"])
    daily, weekly_type = aggregate_student_vle(DATA_DIR / "studentVle.csv", t["vle"], dtypes)
    weekly_total = (weekly_type.groupby(KEYS + ["week"], observed=True)["sum_click"].sum()
                    .reset_index())

    print("[2/6] Building student table ...")
    st = build_student_table(t["studentInfo"], t["studentRegistration"], t["courses"], dtypes)
    print(f"  {len(st):,} enrolments, {st['id_student'].nunique():,} unique students; "
          f"outcome counts:\n{st['final_result'].value_counts().to_string()}")
    st["final_result"] = pd.Categorical(st["final_result"], OUTCOME_ORDER)
    assessments, sassess = t["assessments"], t["studentAssessment"]

    print("[3/6] Temporal visualisations ...")
    W = week_matrix(weekly_total, st, W_MIN, W_MAX, mask_unreg=True)
    plot_weekly_by_group(W, st, "final_result", OUTCOME_ORDER, OUTCOME_PALETTE,
                         "weekly_clicks_by_outcome.png",
                         "Weekly VLE engagement by final result (mean +/- 95% CI)")
    plot_weekly_by_module(W, st, "weekly_clicks_by_module.png")
    age_order = ["0-35", "35-55", "55<="]
    plot_weekly_by_group(W, st, "age_band", age_order,
                         dict(zip(age_order, sns.color_palette("Set2", 3))),
                         "weekly_clicks_by_age.png", "Weekly VLE engagement by age band")

    print("[4/6] Feature engineering (main window) ...")
    df = make_model_table(st, daily, weekly_total, weekly_type, assessments, sassess, EARLY_WEEKS)
    df.to_csv(TAB_DIR / f"model_table_first_{EARLY_WEEKS}_weeks.csv", index=False)
    df["final_result"] = pd.Categorical(df["final_result"], OUTCOME_ORDER)
    plot_activity_mix(df, "activity_mix_by_outcome.png")
    plot_consistency_heatmap(df, "h1_consistency_heatmap.png")
    plot_latency(df, "h2_submission_latency.png")
    run_hypothesis_tests(df)

    print("[5/6] Training models ...")
    run_main_models(df)
    print("[6/6] Ablation + early-warning curve ...")
    run_ablation(df)
    run_early_warning_curve(st, daily, weekly_total, weekly_type, assessments, sassess)
    print(f"\nDone. Figures -> {FIG_DIR}\n      Tables  -> {TAB_DIR}")


if __name__ == "__main__":
    main()

[1/6] Loading tables ...
  chunk 1 processed
  chunk 2 processed
  chunk 3 processed
  chunk 4 processed
  chunk 5 processed
  chunk 6 processed
[2/6] Building student table ...
  32,593 enrolments, 28,785 unique students; outcome counts:
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
[3/6] Temporal visualisations ...
[4/6] Feature engineering (main window) ...
                         coef  odds_ratio  p_value
Intercept              0.8370      2.3093   0.0000
C(code_module)[T.BBB] -0.1303      0.8778   0.2027
C(code_module)[T.CCC] -0.8956      0.4084   0.0000
C(code_module)[T.DDD] -1.3421      0.2613   0.0000
C(code_module)[T.EEE] -0.8388      0.4322   0.0000
C(code_module)[T.FFF] -1.4488      0.2348   0.0000
C(code_module)[T.GGG]  2.6847     14.6534   0.0000
log_total_clicks_z     0.5389      1.7142   0.0000
weekly_cv_z           -0.2420      0.7850   0.0000
active_week_ratio_z    0.3037      1.3549   0.0000
longest_gap_days_z     0.